# 1. Import, configuration and load engineered dataset

In [1]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed"

INPUT_FILE = (
    PROCESSED_DATA_DIR
    / "engineered_air_quality.parquet"
)

TRAIN_FILE = (
    PROCESSED_DATA_DIR
    / "train_air_quality.parquet"
)

VALIDATION_FILE = (
    PROCESSED_DATA_DIR
    / "validation_air_quality.parquet"
)

TEST_FILE = (
    PROCESSED_DATA_DIR
    / "test_air_quality.parquet"
)

SPLIT_METADATA_FILE = (
    PROCESSED_DATA_DIR
    / "split_metadata.json"
)

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

ENCODER_LENGTH = 168
PREDICTION_LENGTH = 15

REQUIRED_SEQUENCE_LENGTH = (
    ENCODER_LENGTH
    + PREDICTION_LENGTH
)

TARGET_COLUMNS = [
    "pm2_5",
    "pm10",
    "no",
    "no2",
    "nox",
    "nh3",
    "co",
    "so2",
    "o3",
]

if not np.isclose(
    TRAIN_RATIO
    + VALIDATION_RATIO
    + TEST_RATIO,
    1.0,
):
    raise ValueError(
        "Train, validation and test ratios "
        "must sum to 1.0."
    )

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Engineered dataset not found: "
        f"{INPUT_FILE}"
    )

df = pd.read_parquet(
    INPUT_FILE,
    engine="pyarrow",
)

required_columns = [
    "timestamp",
    "station_id",
    "time_idx",
    "sequence_segment_id",
] + TARGET_COLUMNS

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{missing_columns}"
    )

df = df.sort_values(
    [
        "station_id",
        "timestamp",
    ],
    kind="mergesort",
).reset_index(drop=True)

print("Engineered dataset loaded successfully.")

print(
    f"Shape: {df.shape}"
)

print(
    f"Rows: {len(df):,}"
)

print(
    f"Stations: "
    f"{df['station_id'].nunique():,}"
)

print(
    f"Timestamp range: "
    f"{df['timestamp'].min()} "
    f"to {df['timestamp'].max()}"
)

print(
    f"Encoder length: "
    f"{ENCODER_LENGTH} hours"
)

print(
    f"Prediction length: "
    f"{PREDICTION_LENGTH} hours"
)

print(
    f"Required sequence length: "
    f"{REQUIRED_SEQUENCE_LENGTH} hours"
)

Engineered dataset loaded successfully.
Shape: (3429120, 38)
Rows: 3,429,120
Stations: 66
Timestamp range: 2017-01-01 00:00:00 to 2026-06-30 23:00:00
Encoder length: 168 hours
Prediction length: 15 hours
Required sequence length: 183 hours


# 2. Calculate global chronological split boundaries

In [2]:
unique_timestamps = (
    df["timestamp"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

timestamp_count = len(
    unique_timestamps
)

train_boundary_index = int(
    timestamp_count
    * TRAIN_RATIO
)

validation_boundary_index = int(
    timestamp_count
    * (
        TRAIN_RATIO
        + VALIDATION_RATIO
    )
)

if (
    train_boundary_index <= 0
    or validation_boundary_index
    <= train_boundary_index
    or validation_boundary_index
    >= timestamp_count
):
    raise ValueError(
        "Invalid chronological split boundaries."
    )

train_end_timestamp = unique_timestamps.iloc[
    train_boundary_index - 1
]

validation_start_timestamp = (
    unique_timestamps.iloc[
        train_boundary_index
    ]
)

validation_end_timestamp = (
    unique_timestamps.iloc[
        validation_boundary_index - 1
    ]
)

test_start_timestamp = unique_timestamps.iloc[
    validation_boundary_index
]

print("GLOBAL CHRONOLOGICAL SPLIT BOUNDARIES")

print(
    f"\nUnique timestamps: "
    f"{timestamp_count:,}"
)

print(
    f"\nTrain:"
    f"\n  Start: {unique_timestamps.iloc[0]}"
    f"\n  End:   {train_end_timestamp}"
)

print(
    f"\nValidation:"
    f"\n  Start: {validation_start_timestamp}"
    f"\n  End:   {validation_end_timestamp}"
)

print(
    f"\nTest:"
    f"\n  Start: {test_start_timestamp}"
    f"\n  End:   {unique_timestamps.iloc[-1]}"
)

print(
    "\nSplit strategy: "
    "global chronological 70/15/15"
)

GLOBAL CHRONOLOGICAL SPLIT BOUNDARIES

Unique timestamps: 83,231

Train:
  Start: 2017-01-01 00:00:00
  End:   2023-08-25 13:00:00

Validation:
  Start: 2023-08-25 14:00:00
  End:   2025-01-26 18:00:00

Test:
  Start: 2025-01-26 19:00:00
  End:   2026-06-30 23:00:00

Split strategy: global chronological 70/15/15


# 3. Create, validation and test split

In [3]:
train_df = df[
    df["timestamp"]
    <= train_end_timestamp
].copy()

validation_df = df[
    (
        df["timestamp"]
        >= validation_start_timestamp
    )
    & (
        df["timestamp"]
        <= validation_end_timestamp
    )
].copy()

test_df = df[
    df["timestamp"]
    >= test_start_timestamp
].copy()

split_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for split_name, split_df in split_frames.items():

    split_df.sort_values(
        [
            "station_id",
            "timestamp",
        ],
        kind="mergesort",
        inplace=True,
    )

    split_df.reset_index(
        drop=True,
        inplace=True,
    )

    if split_df.empty:
        raise ValueError(
            f"{split_name} split is empty."
        )

print("DATASET SPLITS CREATED")

split_summary = pd.DataFrame(
    {
        split_name: {
            "rows": len(split_df),
            "row_percent": (
                len(split_df)
                / len(df)
                * 100
            ),
            "stations": (
                split_df["station_id"]
                .nunique()
            ),
            "segments": (
                split_df[
                    [
                        "station_id",
                        "sequence_segment_id",
                    ]
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "start_timestamp": (
                split_df["timestamp"].min()
            ),
            "end_timestamp": (
                split_df["timestamp"].max()
            ),
        }
        for split_name, split_df
        in split_frames.items()
    }
).T

display(split_summary)

print(
    f"\nTotal split rows: "
    f"{sum(len(frame) for frame in split_frames.values()):,}"
)

print(
    f"Original rows: "
    f"{len(df):,}"
)

DATASET SPLITS CREATED


,rows,row_percent,stations,segments,start_timestamp,end_timestamp
train,2007399,58.5398,66,4795,2017-01-01 00:00:00,2023-08-25 13:00:00
validation,747056,21.7856,65,1995,2023-08-25 14:00:00,2025-01-26 18:00:00
test,674665,19.6746,64,2238,2025-01-26 19:00:00,2026-06-30 23:00:00



Total split rows: 3,429,120
Original rows: 3,429,120


# 4. Validate leakage preventation and sequence readness

In [4]:
def calculate_sequence_readiness(
    split_df,
):
    segment_lengths = (
        split_df.groupby(
            [
                "station_id",
                "sequence_segment_id",
            ],
            observed=True,
            sort=False,
        )
        .size()
    )

    valid_segments = segment_lengths[
        segment_lengths
        >= REQUIRED_SEQUENCE_LENGTH
    ]

    valid_windows = int(
        (
            valid_segments
            - REQUIRED_SEQUENCE_LENGTH
            + 1
        ).sum()
    )

    return {
        "segments": len(segment_lengths),
        "valid_segments": len(valid_segments),
        "valid_windows": valid_windows,
        "stations_with_valid_sequences": (
            valid_segments
            .reset_index()["station_id"]
            .nunique()
        ),
    }


assert (
    train_df["timestamp"].max()
    < validation_df["timestamp"].min()
), (
    "Train-validation temporal overlap detected."
)

assert (
    validation_df["timestamp"].max()
    < test_df["timestamp"].min()
), (
    "Validation-test temporal overlap detected."
)

total_split_rows = sum(
    len(split_df)
    for split_df in split_frames.values()
)

assert total_split_rows == len(df), (
    "Rows were lost or duplicated during splitting."
)

train_indices = set(
    train_df.index
)

validation_indices = set(
    validation_df.index
)

test_indices = set(
    test_df.index
)

for split_name, split_df in split_frames.items():

    duplicate_count = int(
        split_df.duplicated(
            subset=[
                "station_id",
                "timestamp",
            ]
        ).sum()
    )

    if duplicate_count:
        raise ValueError(
            f"{split_name} contains "
            f"{duplicate_count:,} duplicate "
            "station-timestamp rows."
        )

    station_monotonic = (
        split_df.groupby(
            "station_id",
            observed=True,
            sort=False,
        )["timestamp"]
        .apply(
            lambda series:
            series.is_monotonic_increasing
        )
    )

    if not bool(
        station_monotonic.all()
    ):
        raise ValueError(
            f"{split_name} timestamps are "
            "not station-wise chronological."
        )

sequence_readiness = pd.DataFrame(
    {
        split_name:
        calculate_sequence_readiness(
            split_df
        )
        for split_name, split_df
        in split_frames.items()
    }
).T

print("TEMPORAL LEAKAGE CHECK: PASSED")

print(
    "\nTrain end < validation start:",
    train_df["timestamp"].max()
    < validation_df["timestamp"].min(),
)

print(
    "Validation end < test start:",
    validation_df["timestamp"].max()
    < test_df["timestamp"].min(),
)

print(
    "\nSEQUENCE READINESS BY SPLIT"
)

display(sequence_readiness)

if (
    sequence_readiness[
        "valid_windows"
    ]
    .le(0)
    .any()
):
    invalid_splits = (
        sequence_readiness[
            sequence_readiness[
                "valid_windows"
            ]
            <= 0
        ]
        .index
        .tolist()
    )

    raise ValueError(
        "No valid forecasting windows in: "
        f"{invalid_splits}"
    )

print(
    "\nAll splits contain valid "
    "168 → 15 forecasting windows."
)

gc.collect()

TEMPORAL LEAKAGE CHECK: PASSED

Train end < validation start: True
Validation end < test start: True

SEQUENCE READINESS BY SPLIT


,segments,valid_segments,valid_windows,stations_with_valid_sequences
train,4795,1675,1559445,66
validation,1995,694,560498,65
test,2238,642,479204,64



All splits contain valid 168 → 15 forecasting windows.


0

# 5. Save split, metadata and final validation

In [5]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_files = {
    "train": TRAIN_FILE,
    "validation": VALIDATION_FILE,
    "test": TEST_FILE,
}

for split_name, split_df in split_frames.items():

    output_file = output_files[
        split_name
    ]

    split_df.to_parquet(
        output_file,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    if not output_file.exists():
        raise FileNotFoundError(
            f"{split_name} output file "
            "was not created."
        )

    print(
        f"{split_name.capitalize()} saved:"
        f"\n  {output_file}"
        f"\n  Rows: {len(split_df):,}"
        f"\n  Size: "
        f"{output_file.stat().st_size / 1024**2:,.2f} MB"
    )


split_metadata = {
    "split_strategy": (
        "global_chronological"
    ),
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VALIDATION_RATIO,
    "test_ratio": TEST_RATIO,
    "encoder_length": ENCODER_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "required_sequence_length": (
        REQUIRED_SEQUENCE_LENGTH
    ),
    "target_columns": TARGET_COLUMNS,
    "train": {
        "rows": len(train_df),
        "start_timestamp": str(
            train_df["timestamp"].min()
        ),
        "end_timestamp": str(
            train_df["timestamp"].max()
        ),
        "valid_windows": int(
            sequence_readiness.loc[
                "train",
                "valid_windows",
            ]
        ),
    },
    "validation": {
        "rows": len(validation_df),
        "start_timestamp": str(
            validation_df[
                "timestamp"
            ].min()
        ),
        "end_timestamp": str(
            validation_df[
                "timestamp"
            ].max()
        ),
        "valid_windows": int(
            sequence_readiness.loc[
                "validation",
                "valid_windows",
            ]
        ),
    },
    "test": {
        "rows": len(test_df),
        "start_timestamp": str(
            test_df["timestamp"].min()
        ),
        "end_timestamp": str(
            test_df["timestamp"].max()
        ),
        "valid_windows": int(
            sequence_readiness.loc[
                "test",
                "valid_windows",
            ]
        ),
    },
}

with open(
    SPLIT_METADATA_FILE,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        split_metadata,
        metadata_file,
        indent=4,
    )

for split_name, output_file in (
    output_files.items()
):
    saved_split = pd.read_parquet(
        output_file,
        columns=[
            "timestamp",
            "station_id",
            "time_idx",
            "sequence_segment_id",
        ],
        engine="pyarrow",
    )

    assert len(saved_split) == len(
        split_frames[split_name]
    ), (
        f"{split_name} saved row count mismatch."
    )

    del saved_split

print("\n" + "=" * 80)

print("FINAL DATA SPLITTING REPORT")

print("=" * 80)

display(split_summary)

print("\nForecast sequence readiness:")

display(sequence_readiness)

print(
    "\nSplit strategy: "
    "Global chronological 70/15/15"
)

print(
    "Temporal leakage: NONE"
)

print(
    "Random shuffling: DISABLED"
)

print(
    "Station sequence order: PRESERVED"
)

print(
    "Window boundary leakage: PREVENTED "
    "by independent split datasets"
)

print(
    f"\nMetadata saved to: "
    f"{SPLIT_METADATA_FILE}"
)

print(
    "\nData splitting completed successfully."
)

print("=" * 80)

del df
gc.collect()

Train saved:
  ../Data/Processed/train_air_quality.parquet
  Rows: 2,007,399
  Size: 55.84 MB
Validation saved:
  ../Data/Processed/validation_air_quality.parquet
  Rows: 747,056
  Size: 20.99 MB
Test saved:
  ../Data/Processed/test_air_quality.parquet
  Rows: 674,665
  Size: 16.83 MB

FINAL DATA SPLITTING REPORT


,rows,row_percent,stations,segments,start_timestamp,end_timestamp
train,2007399,58.5398,66,4795,2017-01-01 00:00:00,2023-08-25 13:00:00
validation,747056,21.7856,65,1995,2023-08-25 14:00:00,2025-01-26 18:00:00
test,674665,19.6746,64,2238,2025-01-26 19:00:00,2026-06-30 23:00:00



Forecast sequence readiness:


,segments,valid_segments,valid_windows,stations_with_valid_sequences
train,4795,1675,1559445,66
validation,1995,694,560498,65
test,2238,642,479204,64



Split strategy: Global chronological 70/15/15
Temporal leakage: NONE
Random shuffling: DISABLED
Station sequence order: PRESERVED
Window boundary leakage: PREVENTED by independent split datasets

Metadata saved to: ../Data/Processed/split_metadata.json

Data splitting completed successfully.


33